# Bagging intuition with three decision trees

A parcel-routing desk uses two features to decide between **Auto** (continue immediately) and **Review** (send to manual review). We will fit three shallow trees to different bootstrap samples and follow the same new parcel through all three trees.

**Before teaching.** Run all cells. Start the live demonstration at **Draw rows with replacement**. Change `replace=True` to `False`, rerun that cell, and compare the distinct row counts. Restore `True`, rerun, then fit and predict from that sample in the next cell.

This notebook generates its 12 synthetic training rows locally and uses a fixed seed so its worked example matches the interactive teaching page. NumPy, pandas and scikit-learn are the only libraries used.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

print(f'NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

## Set up the parcel desk

`X` contains the two input features per parcel. `y` contains the known route for each training row. The new parcel has measured weight and scan score, and the model will predict its route. `P1` through `P12` are row IDs used to watch the resampling.

In [ ]:
training = pd.DataFrame([
    ('P1', 1.2, 2.0, 0), ('P2', 2.0, 5.0, 0), ('P3', 2.5, 7.8, 0),
    ('P4', 3.5, 4.0, 0), ('P5', 4.0, 7.0, 1), ('P6', 5.0, 5.0, 0),
    ('P7', 6.1, 5.5, 1), ('P8', 7.0, 3.0, 0), ('P9', 7.5, 8.0, 1),
    ('P10', 8.2, 6.6, 1), ('P11', 8.8, 4.5, 1), ('P12', 9.2, 8.8, 1),
], columns=['parcel_id', 'weight', 'scan_score', 'label']).set_index('parcel_id')

features = ['weight', 'scan_score']
X = training[features]
y = training['label']
names = {0: 'Auto', 1: 'Review'}
parcel = pd.DataFrame([(5.65, 5.70)], columns=features, index=['New parcel'])
bootstrap_seed = 11
tree_seed = 0

display(training[features].assign(route=y.map(names)), parcel)

## Draw rows with replacement

A bootstrap sample is a sequence of draws from the original training rows, with every row eligible again on the next draw. The sample size below equals the original dataset size. With 12 draws **without** replacement, all 12 IDs appear once. With replacement, the same row may appear several times and some IDs can be absent.

`rng.choice` draws integer row positions from 0 through 11. `iloc` selects by those positions. The seed makes the worked example repeatable, including when this cell is rerun.

**Worked checkpoint.** The seeded run with replacement has 12 draws and 8 distinct rows. P3, P4, P7, and P11 are omitted. With `replace=False` there are 12 distinct rows. Restore `replace=True` for the rest of the worked example.

In [ ]:
rng = np.random.default_rng(bootstrap_seed)
draw = rng.choice(len(X), size=len(X), replace=True)
drawn_ids = training.index[draw]
counts = drawn_ids.value_counts().reindex(training.index, fill_value=0)

print(f'{len(draw)} draws, {drawn_ids.nunique()} distinct rows')
print('Draw order:', ', '.join(drawn_ids))
print('Omitted:', ', '.join(counts.index[counts == 0]) or 'none')
display(counts.rename('times drawn').to_frame().T)

## Fit one tree and predict one parcel

Each appearance of a repeated row contributes to the fit. All trees use the same learning settings here, including Gini impurity and a maximum depth of two. Point to the first question and then follow the new parcel toward its prediction. For the preset sample, the root compares scan score with 5.80.

In [ ]:
tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
tree.fit(X.iloc[draw], y.iloc[draw])
prediction = int(tree.predict(parcel)[0])

print(f'Tree 1 predicts {names[prediction]} for the new parcel.')
print(export_text(tree, feature_names=features, decimals=2))

## Bootstrap, fit, predict, then combine

A fresh bootstrap sample is drawn before each fit. The sample generator below begins with the same seed so the first draw matches the worked example. Each row of the result gives one tree's class prediction and its leaf-based probability of Review. We place probabilities in columns using `classes_`, which gives each model's class order.

In [ ]:
rng = np.random.default_rng(bootstrap_seed)
demo_draws = rng.choice(len(X), size=(3, len(X)), replace=True)
trees = []
probabilities = np.zeros((len(demo_draws), len(names)))
demo_results = []

for index, sample in enumerate(demo_draws):
    model = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
    model.fit(X.iloc[sample], y.iloc[sample])
    trees.append(model)
    probabilities[index, model.classes_] = model.predict_proba(parcel)[0]
    demo_results.append({
        'tree': index + 1,
        'distinct rows': len(np.unique(sample)),
        'root feature': features[model.tree_.feature[0]],
        'root cut': model.tree_.threshold[0],
        'class': names[int(model.predict(parcel)[0])],
        'P(Review)': probabilities[index, 1],
    })

display(pd.DataFrame(demo_results).round(4))

## Two ways to aggregate a classification prediction

**Hard voting.** Each tree contributes its predicted class. Pick the class with the most votes.

**Probability averaging.** Each tree contributes its probability for each class. Average those numbers class by class and pick the class with the highest mean probability. These rules can yield different outcomes because they use different information. A tie below resolves to the smallest encoded class.

**Worked checkpoint.** In tree order, the predicted classes are Auto, Review, and Review. The corresponding Review probabilities are 0, 1, and 2/3. Review wins the hard vote 2 to 1 and also wins by the mean Review probability of 5/9, about 55.6%.

**Regression.** When trees predict a numerical target, bagging can average their numerical predictions directly.

In [ ]:
hard_predictions = np.array([model.predict(parcel)[0] for model in trees])
votes = np.bincount(hard_predictions, minlength=len(names))
mean_probabilities = probabilities.mean(axis=0)

print(f'Hard vote: Auto {votes[0]}, Review {votes[1]} -> {names[int(votes.argmax())]}')
print(f'Mean P(Review): {mean_probabilities[1]:.4f} -> {names[int(mean_probabilities.argmax())]}')
np.testing.assert_array_equal(hard_predictions, [0, 1, 1])
np.testing.assert_allclose(probabilities[:, 1], [0, 1, 2 / 3])

## Connect the example to Random Forest

Random Forest can use a bootstrap sample per tree. It also restricts the candidate features considered at individual splits. Here `max_features=1` asks the splitter to consider one candidate feature at a time, with additional candidates inspected if needed to find a valid split. Each fitted forest tree returns class probabilities. Scikit-learn averages them and selects the class with the highest average.

This fit has its own randomized trees and probability. With scikit-learn 1.6.1, the 64-tree example predicts Review with mean probability 0.6873. The manual three-tree example above uses an explicit set of draws so each tree can be inspected.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=64, max_depth=2, max_features=1,
    bootstrap=True, random_state=bootstrap_seed, n_jobs=1,
).fit(X, y)

# This example encodes the two classes as 0 and 1, in the same column order.
parcel_array = parcel.to_numpy()
tree_probabilities = np.array([estimator.predict_proba(parcel_array)[0]
                               for estimator in forest.estimators_])
forest_probabilities = forest.predict_proba(parcel)[0]
np.testing.assert_allclose(tree_probabilities.mean(axis=0), forest_probabilities)
display(pd.Series(forest_probabilities, index=[names[int(c)] for c in forest.classes_],
                  name='Mean forest probability').round(4).to_frame().T)
print('Random Forest prediction:', names[int(forest.predict(parcel)[0])])